In [9]:
!pip install -q transformers datasets pandas pyarrow torch evaluate accelerate

In [10]:
import pandas as pd
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)
print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [11]:
# ================================================
# Block 3: Load the FULL dataset from Hugging Face (NO UPLOAD NEEDED!)
# ================================================
print("🌐 Loading full SemEval-2026 Task 13 dataset from Hugging Face...")

full_dataset = load_dataset("DaniilOr/SemEval-2026-Task13","A")

print("Available splits:", list(full_dataset.keys()))
print("Columns:", full_dataset["train"].column_names)

# Keep only what we need for Subtask A (binary AI detection)
train_dataset = full_dataset["train"].select_columns(['code', 'label'])
val_dataset   = full_dataset["validation"].select_columns(['code', 'label'])

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print(f"✅ Full dataset loaded!")
print(f"   Train samples : {len(dataset['train']):,}")
print(f"   Val samples   : {len(dataset['validation']):,}")
print("\nSample row:")
print(dataset["train"][0])

🌐 Loading full SemEval-2026 Task 13 dataset from Hugging Face...
Available splits: ['train', 'validation', 'test']
Columns: ['code', 'generator', 'label', 'language']
✅ Full dataset loaded!
   Train samples : 500,000
   Val samples   : 100,000

Sample row:
{'code': "(a, b, c, d) = [int(x) for x in input().split()]\nk = input()\n(p, q, r, s) = (0, 0, 0, 0)\nfor i in k:\n\tif i == '1':\n\t\tp += 1\n\telif i == '2':\n\t\tq += 1\n\telif i == '3':\n\t\tr += 1\n\telif i == '4':\n\t\ts += 1\nprint(a * p + b * q + c * r + d * s)\n", 'label': 0}


In [12]:


# ================================================
# Block 4: Tokenization
# ================================================
model_name = "microsoft/unixcoder-base"   # This is the official UniXcoder model (you wrote UniXCode)

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # UniXcoder works great on raw code (including newlines)
    return tokenizer(
        examples["code"],
        truncation=True,
        max_length=512,
    )

# Apply tokenization (batched = faster)
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["code"],   # we don't need raw text anymore
    num_proc=1,
    desc="Tokenizing"
)

# Set format for PyTorch
tokenized_datasets.set_format("torch")

print("✅ Tokenization completed")


Tokenizing:   0%|          | 0/100000 [00:00<?, ? examples/s]

✅ Tokenization completed


In [13]:
# ================================================
# Block 5: Load UniXcoder model with classification head
# ================================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,                 # binary: 0=human, 1=AI
    problem_type="single_label_classification"
)

print("✅ Model loaded")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded


In [14]:
# ================================================
# Block 6: Define metrics (Accuracy + Macro F1 - standard for detection tasks)
# ================================================
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {
        "accuracy": acc,
        "f1_macro": f1
    }

In [15]:

# ================================================
# Block 7: Training arguments (baseline settings)
# ================================================
training_args = TrainingArguments(
    output_dir="unixcoder-ai-code-detection-baseline",
    eval_strategy="steps",
    eval_steps=1500, # 0.25 epochs (500,000 samples / 8 batch_size * 0.25)
    save_strategy="steps",
    save_steps=1500, # Save checkpoints at the same frequency as evaluation
    learning_rate=2e-5,
    per_device_train_batch_size=32,      # adjust based on your GPU (16GB+ recommended)
    per_device_eval_batch_size=32,
    num_train_epochs=3,                 # baseline = 3 epochs
    weight_decay=0.01,
    logging_steps=1500,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to=["none"],         # Report to TensorBoard for log files
    push_to_hub=False,
    fp16=True,                          # faster on GPU
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)


In [16]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# ================================================
# Block 8: Create Trainer
# ================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("✅ Trainer ready - starting fine-tuning...")

✅ Trainer ready - starting fine-tuning...


In [17]:

# ================================================
# Block 9: Train the model
# ================================================
trainer.train()



Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ================================================
# Block 10: Final evaluation + save
# ================================================
eval_results = trainer.evaluate()
print("\n=== FINAL EVALUATION ===")
print(f"Accuracy : {eval_results['eval_accuracy']:.4f}")
print(f"F1-macro  : {eval_results['eval_f1_macro']:.4f}")

# Save model & tokenizer
trainer.save_model("unixcoder-ai-code-detection-final")
tokenizer.save_pretrained("unixcoder-ai-code-detection-final")

print("\n🎉 Training finished! Model saved to 'unixcoder-ai-code-detection-final/'")
print("You can now use it for inference or submit to the SemEval leaderboard.")